In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import torch
import gpytorch
from tqdm.auto import tqdm
from linear_operator import settings
from gpytorch import settings as gsettings
from grf_gp.sampler import GRFSampler
from grf_gp.utils.spectral import get_normalized_laplacian
from grf_gp.kernels.diffusion import DiffusionGRFKernel
from grf_gp.kernels.general import GeneralGRFKernel
from grf_gp.model import GRFGP
from grf_gp.utils.config import set_gp_defaults

project_root = os.path.abspath('../../..')
sys.path.append(project_root)
sys.path.append(os.path.abspath('.'))
from data_utils import prepare_wind_graph_data

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_gp_defaults(settings, gsettings)

In [ ]:
NC_FILE = os.path.join(project_root, 'data/wind_interpolation/aac1241c2ceff963a460829f1c68b132.nc')
USE_DOWNSAMPLING = True
DOWNSAMPLE_FACTOR = 20
MAX_WALK_LENGTH = 5
WALKS_PER_NODE = 2048
P_HALT = 0.1
LR = 0.05
MAX_ITER = 300
SEED = 0

In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)

data = prepare_wind_graph_data(
    NC_FILE,
    use_downsampling=USE_DOWNSAMPLING,
    downsample_factor=DOWNSAMPLE_FACTOR,
    dtype=np.float64,
)

X_train = torch.tensor(data['X_train']).long().to(device)
Y_train = torch.tensor(data['y_train'], dtype=torch.float64).flatten().to(device)
X_test = torch.tensor(data['X_test']).long().to(device)
Y_test = torch.tensor(data['y_test'], dtype=torch.float64).flatten().to(device)
X_full = torch.tensor(data['X']).long().to(device)

adjacency_matrix = data['adjacency'].toarray()
L = get_normalized_laplacian(adjacency_matrix)
L = torch.tensor(L, dtype=torch.float64).to(device)
orig_std = float(data['y_std'])

In [ ]:
def train_model(model, likelihood, x_train, y_train, lr=0.01, max_iter=100):
    model.train(); likelihood.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
    losses = []
    pbar = tqdm(total=max_iter, desc='Training', leave=False)
    for _ in range(max_iter):
        optimizer.zero_grad()
        out = model(x_train)
        loss = -mll(out, y_train)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        pbar.update(1)
    pbar.close()
    return losses

def evaluate_model(model, likelihood, x_train, y_train, x_test, y_test, orig_std):
    model.eval(); likelihood.eval()
    with torch.no_grad():
        mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
        train_output = model(x_train)
        lml = mll(train_output, y_train) * y_train.size(0)
        pred = model.predict(x_test)
        mean = pred.mean
        rmse = orig_std * torch.sqrt(torch.mean((y_test.flatten() - mean.flatten())**2))
        nlpd = -pred.log_prob(y_test.flatten())
    return lml.item(), rmse.item(), nlpd.item()

In [ ]:
sampler = GRFSampler(L, WALKS_PER_NODE, P_HALT, MAX_WALK_LENGTH, seed=SEED, use_tqdm=True, n_processes=4)
rw_mats = sampler.sample_random_walk_matrices()

In [ ]:
likelihood_general = gpytorch.likelihoods.GaussianLikelihood().to(device)
kernel_general = GeneralGRFKernel(MAX_WALK_LENGTH, rw_mats).to(device)
model_general = GRFGP(X_train, Y_train, likelihood_general, kernel_general).to(device)
_ = train_model(model_general, likelihood_general, X_train, Y_train, lr=LR, max_iter=MAX_ITER)
lml_g, rmse_g, nlpd_g = evaluate_model(model_general, likelihood_general, X_train, Y_train, X_test, Y_test, orig_std)
print(f'GeneralGRFKernel | LML: {lml_g:.4f} | RMSE: {rmse_g:.4f} | NLPD: {nlpd_g:.4f}')
print('modulation:', kernel_general.modulation_function.detach().cpu().numpy())

In [ ]:
likelihood_diff = gpytorch.likelihoods.GaussianLikelihood().to(device)
kernel_diff = DiffusionGRFKernel(MAX_WALK_LENGTH, rw_mats).to(device)
model_diff = GRFGP(X_train, Y_train, likelihood_diff, kernel_diff).to(device)
_ = train_model(model_diff, likelihood_diff, X_train, Y_train, lr=LR, max_iter=MAX_ITER)
lml_d, rmse_d, nlpd_d = evaluate_model(model_diff, likelihood_diff, X_train, Y_train, X_test, Y_test, orig_std)
print(f'DiffusionGRFKernel | LML: {lml_d:.4f} | RMSE: {rmse_d:.4f} | NLPD: {nlpd_d:.4f}')
print(f'beta: {kernel_diff.beta.item():.4f}, sigma_f: {kernel_diff.sigma_f.item():.4f}')